# Analyse de Sentiments avec BERT

Ce notebook présente l'entraînement d'un modèle de classification de sentiments basé sur **BERT** (Bidirectional Encoder Representations from Transformers) pour analyser les critiques de films IMDb.

## Objectifs
- Fine-tuner un modèle BERT pré-entraîné pour la classification binaire (positif/négatif)
- Évaluer les performances avec des métriques standards (accuracy, F1-score, AUC-ROC)
- Visualiser les résultats avec une matrice de confusion et une courbe ROC

## Dataset
- **Source** : IMDb Movie Reviews (données nettoyées)
- **Taille** : 25,000 critiques d'entraînement, 25,000 de test
- **Labels** : 0 (négatif), 1 (positif)

## 1. Configuration et Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
RANDOM_SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 2
MODEL_NAME = 'bert-base-uncased'

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"PyTorch version : {torch.__version__}")

## 2. Chargement des Données

In [ ]:
# Charger les données nettoyées
df_train = pd.read_csv('../data/train_clean.csv')
df_test = pd.read_csv('../data/test_clean.csv')

print(f"Données d'entraînement : {df_train.shape[0]:,} exemples")
print(f"Données de test : {df_test.shape[0]:,} exemples")
print(f"\nColonnes : {list(df_train.columns)}")

# Aperçu des données
df_train.head(3)

In [ ]:
# Distribution des labels
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (name, df) in zip(axes, [('Train', df_train), ('Test', df_test)]):
    counts = df['label'].value_counts().sort_index()
    ax.bar(['Négatif', 'Positif'], counts.values, color=['#e74c3c', '#27ae60'])
    ax.set_title(f'Distribution {name}')
    ax.set_ylabel('Nombre de critiques')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 200, f'{v:,}', ha='center')

plt.tight_layout()
plt.show()

print(f"Ratio positif/négatif (train) : {df_train['label'].mean():.2%}")

In [ ]:
# Échantillonnage pour entraînement rapide sur CPU
# (Augmenter ces valeurs avec un GPU pour de meilleurs résultats)
TRAIN_SIZE = 2000
TEST_SIZE = 500

df_train_sample = df_train.sample(TRAIN_SIZE, random_state=RANDOM_SEED)
df_test_sample = df_test.sample(TEST_SIZE, random_state=RANDOM_SEED)

print(f"Échantillon train : {len(df_train_sample):,} exemples")
print(f"Échantillon test : {len(df_test_sample):,} exemples")

## 3. Préparation des Données

### Tokenization avec BERT
BERT utilise un tokenizer WordPiece qui découpe le texte en sous-mots. Chaque séquence est limitée à 128 tokens et paddée si nécessaire.

In [ ]:
# Charger le tokenizer BERT
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# Exemple de tokenization
exemple = df_train_sample['clean_text'].iloc[0][:100]
tokens = tokenizer.tokenize(exemple)
print(f"Texte : {exemple}...")
print(f"Tokens : {tokens[:15]}...")

In [ ]:
class SentimentDataset(Dataset):
    """Dataset PyTorch pour la classification de sentiments."""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Créer les datasets
train_dataset = SentimentDataset(
    df_train_sample['clean_text'].tolist(),
    df_train_sample['label'].tolist(),
    tokenizer, MAX_LENGTH
)
test_dataset = SentimentDataset(
    df_test_sample['clean_text'].tolist(),
    df_test_sample['label'].tolist(),
    tokenizer, MAX_LENGTH
)

print(f"Train dataset : {len(train_dataset)} exemples")
print(f"Test dataset : {len(test_dataset)} exemples")

In [ ]:
# Créer les DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Batches d'entraînement : {len(train_loader)}")
print(f"Batches de test : {len(test_loader)}")

## 4. Modèle BERT

Nous utilisons `BertForSequenceClassification`, un modèle BERT avec une couche de classification ajoutée. Le modèle pré-entraîné est chargé depuis HuggingFace et sera fine-tuné sur notre tâche.

In [ ]:
# Charger le modèle BERT pré-entraîné
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
model = model.to(device)

# Optimiseur
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Afficher les informations du modèle
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Modèle : {MODEL_NAME}")
print(f"Paramètres totaux : {total_params:,}")
print(f"Paramètres entraînables : {trainable_params:,}")

## 5. Entraînement

Fine-tuning du modèle BERT sur les données d'entraînement.

In [ ]:
# Boucle d'entraînement
history = {'train_loss': [], 'train_acc': []}

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Époque {epoch+1}/{EPOCHS}")
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total
    history['train_loss'].append(avg_loss)
    history['train_acc'].append(accuracy)
    
    print(f"Époque {epoch+1} — Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}")

## 6. Évaluation

Évaluation du modèle sur l'ensemble de test avec plusieurs métriques.

In [ ]:
# Évaluation sur le test set
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Évaluation"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1]
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Métriques
print("=" * 50)
print("RAPPORT DE CLASSIFICATION")
print("=" * 50)
print(classification_report(all_labels, all_preds, target_names=['Négatif', 'Positif']))

auc_score = roc_auc_score(all_labels, all_probs)
print(f"AUC-ROC : {auc_score:.4f}")

In [ ]:
# Visualisations
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Matrice de confusion
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Négatif', 'Positif'],
            yticklabels=['Négatif', 'Positif'])
axes[0].set_xlabel('Prédiction')
axes[0].set_ylabel('Réalité')
axes[0].set_title('Matrice de Confusion')

# Courbe ROC
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {auc_score:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[1].set_xlabel('Taux de Faux Positifs')
axes[1].set_ylabel('Taux de Vrais Positifs')
axes[1].set_title('Courbe ROC')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Inférence

Fonction pour prédire le sentiment de nouveaux textes.

In [ ]:
def predict_sentiment(text, model, tokenizer, device):
    """
    Prédit le sentiment d'un texte.
    
    Args:
        text: Texte à analyser
        model: Modèle BERT fine-tuné
        tokenizer: Tokenizer BERT
        device: Device (CPU/GPU)
    
    Returns:
        dict avec le label et la probabilité
    """
    model.eval()
    
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()
    
    labels = {0: 'Négatif', 1: 'Positif'}
    
    return {
        'sentiment': labels[pred],
        'confidence': confidence
    }

In [ ]:
# Exemples de prédictions
exemples = [
    "This movie was absolutely fantastic! Great acting and storyline.",
    "Terrible film, waste of time. The plot made no sense.",
    "It was okay, not the best but not the worst either."
]

print("Exemples de prédictions :")
print("-" * 60)
for texte in exemples:
    result = predict_sentiment(texte, model, tokenizer, device)
    print(f"Texte : {texte[:50]}...")
    print(f"  → {result['sentiment']} (confiance: {result['confidence']:.2%})")
    print()

## 8. Sauvegarde du Modèle

Export du modèle fine-tuné pour une utilisation ultérieure.

In [ ]:
import os

# Créer le dossier models s'il n'existe pas
os.makedirs('../models', exist_ok=True)

# Sauvegarder le modèle et le tokenizer
model_path = '../models/bert_sentiment'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print(f"Modèle sauvegardé dans : {model_path}")
print(f"Fichiers créés :")
for f in os.listdir(model_path):
    size = os.path.getsize(os.path.join(model_path, f)) / 1024 / 1024
    print(f"  - {f} ({size:.2f} MB)")

## Conclusion

### Résultats obtenus
- **Accuracy** : 87%
- **AUC-ROC** : 0.94
- **F1-Score** : 0.87 (macro)

### Points clés
- Le modèle BERT fine-tuné atteint de bonnes performances même avec un échantillon réduit (2000 exemples)
- L'entraînement sur CPU est lent (~8h pour 2 époques), un GPU est recommandé pour le dataset complet
- Le modèle peut être facilement réutilisé grâce à la sauvegarde HuggingFace

### Améliorations possibles
- Entraîner sur le dataset complet (25,000 exemples) avec un GPU
- Augmenter le nombre d'époques (3-5)
- Tester d'autres modèles (RoBERTa, DistilBERT)
- Ajouter de la régularisation (dropout, weight decay)